# Práctica 2: Árboles de Decisión con Datos de Rendimiento Académico

**Machine Learning 1 — FIUNA**

**Docente:** Diego Stalder | **Práctica:** Carlos Benítez

---

## Objetivo

Aplicar **Árboles de Decisión** al dataset académico (Ecuaciones Diferenciales → Cálculo 2) para predecir la variable **`Remontada`**, comparando con los modelos anteriores.

## Estructura

1. **Carga y preparación** del dataset académico
2. **Árbol con configuración por defecto** (baseline)
3. **Visualización del árbol** e interpretación de reglas
4. **Análisis de hiperparámetros** (profundidad, min_samples_leaf)
5. **Post-poda** con Cost Complexity Pruning
6. **Importancia de características**
7. **Comparación con modelos anteriores** (LogReg, SVM, SVD+SVM)
8. **Ejercicio final**: Random Forest como mejora natural

## ¿Por qué árboles para datos académicos?

- **Interpretabilidad**: podemos explicar las decisiones en términos de notas y asistencia
- **Sin escalado**: no requiere normalizar las notas
- **Manejo de no linealidades**: las reglas de aprobación suelen ser no lineales
- **Base para Random Forest**: modelo líder en datos tabulares

## 0. Importaciones y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100
})
sns.set_palette('husl')
np.random.seed(42)

print("="*80)
print("🌳 ÁRBOLES DE DECISIÓN CON DATOS DE RENDIMIENTO ACADÉMICO")
print("🎯 Predicción de 'Remontada' en Cálculo 2")
print("="*80)

## 1. Carga y Preparación del Dataset

Usamos el mismo pipeline de la práctica anterior: Ecuaciones Diferenciales como materia previa, Cálculo 2 como materia siguiente.

In [ ]:
# Cargar datos
df = pd.read_csv(r'D:\ArchivosHDD\GitHub\Machine-Learning\Clase_4\datos_concatenados.csv')
print(f"✅ Datos cargados: {len(df):,} registros")

# Filtrar Ecuaciones Diferenciales
mask_ed = df['Asignatura'].str.upper().str.contains('ECUACIONES DIFERENCIALES', na=False)
df_ed = df[mask_ed].copy()

# Filtrar Cálculo 2
mask_c2 = df['Asignatura'].str.upper().str.contains('CALCULO 2', na=False)
df_c2 = df[mask_c2].copy()

# Definir variable objetivo 'Remontada'
df_ed['Remontada'] = (df_ed['Segundo.Par'] > df_ed['Primer.Par']) & (df_ed['Firma'] > 50)

# Merge por alumno
merged = df_ed.merge(df_c2, on='ALUMNO_ID', how='inner', suffixes=('_ed', '_c2'))
print(f"Registros después del merge: {len(merged):,}")

# Features
feature_cols = [
    'Primer.Par_ed', 'Segundo.Par_ed', 'Tercer.Par_ed',
    'TPLab._ed', 'Lab._ed', 'Proy._ed',
    'Asis_ed', 'Requisito_ed', 'Firma_ed',
    'Primer.Rec_ed', 'Segundo.Rec_ed',
]

X = merged[feature_cols].copy()
y = merged['Remontada'].astype(int)

# Convertir a numérico y limpiar
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')

mask_valid = ~X.isnull().any(axis=1)
X = X[mask_valid].reset_index(drop=True)
y = y[mask_valid].reset_index(drop=True)

print(f"\n📊 Dataset final: {len(X)} estudiantes, {X.shape[1]} características")
print(f"Tasa de Remontada: {y.mean()*100:.1f}%")
print(f"\nDistribución de clases:")
print(y.value_counts())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nEntrenamiento: {len(X_train)} | Prueba: {len(X_test)}")

## 2. Árbol con Configuración por Defecto (Baseline)

Entrenamos un árbol sin restricciones para ver su comportamiento inicial.

In [ ]:
# Árbol sin restricciones
tree_full = DecisionTreeClassifier(random_state=42, class_weight='balanced')
tree_full.fit(X_train, y_train)

# Métricas
acc_train = tree_full.score(X_train, y_train)
acc_test = tree_full.score(X_test, y_test)

print("="*80)
print("🌳 ÁRBOL SIN RESTRICCIONES (baseline)")
print("="*80)
print(f"Profundidad alcanzada: {tree_full.get_depth()}")
print(f"Número de hojas:       {tree_full.get_n_leaves()}")
print(f"Accuracy en Train:     {acc_train*100:.2f}%")
print(f"Accuracy en Test:      {acc_test*100:.2f}%")
print(f"Diferencia (Sobreajuste): {(acc_train - acc_test)*100:.2f}%")
print(f"\n⚠️  El árbol sobreajusta: la brecha Train-Test es grande")

In [ ]:
# Predicciones y reporte
y_pred_full = tree_full.predict(X_test)
print(classification_report(y_test, y_pred_full, target_names=['No Remontada', 'Remontada']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_full)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Remontada', 'Remontada'],
            yticklabels=['No Remontada', 'Remontada'])
plt.title('Matriz de Confusión — Árbol sin restricciones', fontweight='bold')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.show()

## 3. Visualización del Árbol e Interpretación

El árbol sin restricciones es demasiado grande para visualizarlo. Veamos uno con profundidad limitada para extraer reglas interpretables.

In [ ]:
# Árbol con profundidad 3 (interpretable)
tree_viz = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
tree_viz.fit(X_train, y_train)

print(f"Árbol de profundidad 3:")
print(f"   Accuracy Train: {tree_viz.score(X_train, y_train)*100:.2f}%")
print(f"   Accuracy Test:  {tree_viz.score(X_test, y_test)*100:.2f}%")
print(f"   Hojas: {tree_viz.get_n_leaves()}")

# Visualización
plt.figure(figsize=(20, 10))
plot_tree(tree_viz, filled=True, feature_names=feature_cols,
          class_names=['No Remontada', 'Remontada'], fontsize=10, rounded=True)
plt.title('Árbol de Decisión (profundidad 3) — Reglas de Remontada', fontweight='bold', fontsize=14)
plt.show()

# Reglas en texto
print("\n" + "="*80)
print("📋 REGLAS DE DECISIÓN EXTRAÍDAS")
print("="*80)
print(export_text(tree_viz, feature_names=feature_cols, max_depth=3))

### 3.1 Interpretación de las Reglas

**Reglas típicas** (los valores exactos se ven en la salida):

- Si **Segundo Parcial > 40** y **Firma > 50** → probablemente **Remontada**
- Si **Primer Parcial > 70** pero **Segundo Parcial < 30** → probablemente **No Remontada**
- Si **Firma < 40** → probablemente **No Remontada** (el estudiante abandonó)

**Conclusión:** La variable `Remontada` depende fundamentalmente de la **mejora entre parciales** (que ya está incorporada indirectamente) y de la **firma** (compromiso del estudiante).

## 4. Análisis de Hiperparámetros

Exploramos el efecto de `max_depth` y `min_samples_leaf` en el sobreajuste.

In [ ]:
# Curva de validación: max_depth
profundidades = range(1, 25)
train_scores, test_scores = [], []

for d in profundidades:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42, class_weight='balanced')
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

best_d = profundidades[np.argmax(test_scores)]

plt.figure(figsize=(12, 6))
plt.plot(profundidades, train_scores, 'o-', color='#2980B9', linewidth=2, label='Train')
plt.plot(profundidades, test_scores, 's-', color='#C0392B', linewidth=2, label='Test')
plt.axvline(x=best_d, color='green', linestyle='--', alpha=0.7, label=f'Óptimo: {best_d}')
plt.xlabel('Profundidad máxima', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Curva de Validación: Profundidad del Árbol', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"🌳 Mejor profundidad (por accuracy en test): {best_d}")
print(f"   Train accuracy: {train_scores[best_d-1]*100:.2f}%")
print(f"   Test accuracy:  {test_scores[best_d-1]*100:.2f}%")

In [ ]:
# GridSearchCV con pre-poda completa
param_grid = {
    'max_depth': [2, 3, 4, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20, 30],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None]
}

tree_grid = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(tree_grid, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
t0 = time.time()
grid.fit(X_train, y_train)
t_grid = time.time() - t0

print("="*80)
print("🏆 GRIDSEARCHCV: MEJOR CONFIGURACIÓN DE PRE-PODA")
print("="*80)
print(f"Tiempo: {t_grid:.2f}s")
print(f"Mejores hiperparámetros: {grid.best_params_}")
print(f"Mejor AUC en CV:         {grid.best_score_:.4f}")

# Evaluar en test
y_pred_grid = grid.predict(X_test)
y_prob_grid = grid.predict_proba(X_test)[:, 1]
acc_grid = accuracy_score(y_test, y_pred_grid)
auc_grid = roc_auc_score(y_test, y_prob_grid)

print(f"\nAccuracy en test: {acc_grid*100:.2f}%")
print(f"AUC-ROC en test:  {auc_grid:.4f}")
print(f"Hojas del mejor árbol: {grid.best_estimator_.get_n_leaves()}")

## 5. Post-poda: Cost Complexity Pruning

Aplicamos CCP para encontrar el árbol más simple que mantiene buen desempeño.

In [ ]:
# Determinar alpha óptimo
tree_for_path = DecisionTreeClassifier(random_state=42, class_weight='balanced')
path = tree_for_path.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]

print(f"Alphas generados: {len(alphas)}")
print(f"Rango de alpha: [{alphas.min():.6f}, {alphas.max():.6f}]")

# Probar alphas con CV
cv_scores = []
n_leaves_list = []

for alpha in alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42, class_weight='balanced')
    scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='roc_auc')
    cv_scores.append(scores.mean())
    tree.fit(X_train, y_train)
    n_leaves_list.append(tree.get_n_leaves())

best_idx = np.argmax(cv_scores)
best_alpha = alphas[best_idx]

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(alphas, cv_scores, 'o-', color='#2980B9', markersize=4)
axes[0].axvline(x=best_alpha, color='green', linestyle='--', label=f'Óptimo: {best_alpha:.5f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('ccp_alpha')
axes[0].set_ylabel('AUC-ROC (CV)')
axes[0].set_title('Post-poda: AUC-ROC vs Alpha', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(alphas, n_leaves_list, 's-', color='#F39C12', markersize=4)
axes[1].axvline(x=best_alpha, color='green', linestyle='--')
axes[1].set_xscale('log')
axes[1].set_xlabel('ccp_alpha')
axes[1].set_ylabel('Número de hojas')
axes[1].set_title('Complejidad del árbol vs Alpha', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Entrenar árbol podado
tree_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42, class_weight='balanced')
tree_pruned.fit(X_train, y_train)

print(f"\n🌳 Árbol podado:")
print(f"   Alpha óptimo: {best_alpha:.6f}")
print(f"   Profundidad: {tree_pruned.get_depth()}")
print(f"   Hojas: {tree_pruned.get_n_leaves()}")
print(f"   Accuracy Train: {tree_pruned.score(X_train, y_train)*100:.2f}%")
print(f"   Accuracy Test:  {tree_pruned.score(X_test, y_test)*100:.2f}%")
print(f"   AUC Test:       {roc_auc_score(y_test, tree_pruned.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
# Visualizar árbol podado
plt.figure(figsize=(20, 10))
plot_tree(tree_pruned, filled=True, feature_names=feature_cols,
          class_names=['No Remontada', 'Remontada'], fontsize=10, rounded=True)
plt.title(f'Árbol Podado (α={best_alpha:.5f}) — Reglas Simplificadas', fontweight='bold', fontsize=14)
plt.show()

## 6. Importancia de Características

El árbol revela qué características son más importantes para predecir `Remontada`.

In [ ]:
# Importancias del árbol completo
importancias = pd.DataFrame({
    'Característica': feature_cols,
    'Importancia': tree_full.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#C0392B' if imp > 0.15 else ('#F39C12' if imp > 0.05 else '#BDC3C7')
          for imp in importancias['Importancia']]
plt.barh(importancias['Característica'], importancias['Importancia'], color=colors, edgecolor='k')
plt.xlabel('Importancia', fontsize=12)
plt.title('Importancia de Características — Predicción de Remontada', fontweight='bold')
plt.grid(alpha=0.3, axis='x')
plt.show()

print("="*80)
print("📊 IMPORTANCIA DE CARACTERÍSTICAS")
print("="*80)
print(importancias.sort_values('Importancia', ascending=False).to_string(index=False))

print("\n💡 Interpretación:")
print("   - Características con importancia > 0.15: muy relevantes")
print("   - Características con importancia < 0.05: poco relevantes")

## 7. Comparación con Modelos Anteriores

Comparamos el árbol con todos los modelos que hemos construido en clases anteriores.

In [ ]:
# Regresión Logística
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])
pipe_lr.fit(X_train, y_train)
acc_lr = pipe_lr.score(X_test, y_test)
auc_lr = roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])

# SVM RBF
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
                random_state=42, probability=True))
])
pipe_svm.fit(X_train, y_train)
acc_svm = pipe_svm.score(X_test, y_test)
auc_svm = roc_auc_score(y_test, pipe_svm.predict_proba(X_test)[:, 1])

# Árbol sin poda
acc_tree_full = acc_test
auc_tree_full = roc_auc_score(y_test, tree_full.predict_proba(X_test)[:, 1])

# Árbol con pre-poda
acc_tree_grid = acc_grid
auc_tree_grid = auc_grid

# Árbol con post-poda
acc_tree_pruned = tree_pruned.score(X_test, y_test)
auc_tree_pruned = roc_auc_score(y_test, tree_pruned.predict_proba(X_test)[:, 1])

# Tabla comparativa
comparacion = pd.DataFrame({
    'Modelo': [
        'Regresión Logística',
        'SVM RBF',
        'Árbol (sin poda)',
        'Árbol (pre-poda GridSearch)',
        'Árbol (post-poda CCP)'
    ],
    'Accuracy': [acc_lr, acc_svm, acc_tree_full, acc_tree_grid, acc_tree_pruned],
    'AUC-ROC': [auc_lr, auc_svm, auc_tree_full, auc_tree_grid, auc_tree_pruned]
})
comparacion['Accuracy'] = (comparacion['Accuracy']*100).round(2)
comparacion['AUC-ROC'] = comparacion['AUC-ROC'].round(4)

print("="*80)
print("📊 COMPARACIÓN DE MODELOS")
print("="*80)
print(comparacion.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#3498DB', '#9B59B6', '#E67E22', '#27AE60', '#C0392B']

axes[0].barh(comparacion['Modelo'], comparacion['Accuracy'], color=colors, edgecolor='k')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Accuracy en Test', fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')
for i, v in enumerate(comparacion['Accuracy']):
    axes[0].text(v+0.5, i, f'{v:.1f}%', va='center')

axes[1].barh(comparacion['Modelo'], comparacion['AUC-ROC'], color=colors, edgecolor='k')
axes[1].set_xlabel('AUC-ROC')
axes[1].set_title('AUC-ROC en Test', fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')
for i, v in enumerate(comparacion['AUC-ROC']):
    axes[1].text(v+0.005, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC comparativas
plt.figure(figsize=(10, 7))

modelos = [
    ('Regresión Logística', pipe_lr, '#3498DB'),
    ('SVM RBF', pipe_svm, '#9B59B6'),
    ('Árbol (post-poda)', tree_pruned, '#C0392B'),
]

for nombre, modelo, color in modelos:
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{nombre} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio')
plt.xlabel('Tasa de Falsos Positivos', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos', fontsize=12)
plt.title('Curvas ROC — Comparación de Modelos', fontweight='bold', fontsize=13)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

## 8. Bonus: Random Forest (Mejora Natural del Árbol)

Un solo árbol tiene **alta varianza**. Random Forest entrena **muchos árboles** sobre subconjuntos aleatorios de datos y características, y promedia sus predicciones.

**Idea clave:** combinar muchos árboles débiles → un modelo fuerte.

In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

acc_rf = rf.score(X_test, y_test)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print("="*80)
print("🌲 RANDOM FOREST (200 árboles)")
print("="*80)
print(f"Accuracy Train: {rf.score(X_train, y_train)*100:.2f}%")
print(f"Accuracy Test:  {acc_rf*100:.2f}%")
print(f"AUC-ROC Test:   {auc_rf:.4f}")

# Comparar con árbol individual
print(f"\n📊 Comparación con árbol individual:")
print(f"   Árbol podado:    Accuracy {acc_tree_pruned*100:.2f}%, AUC {auc_tree_pruned:.4f}")
print(f"   Random Forest:   Accuracy {acc_rf*100:.2f}%, AUC {auc_rf:.4f}")
print(f"   Mejora:          +{(acc_rf-acc_tree_pruned)*100:.2f}% accuracy, +{auc_rf-auc_tree_pruned:.4f} AUC")

# Importancias del Random Forest
importancias_rf = pd.DataFrame({
    'Característica': feature_cols,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importancias_rf['Característica'], importancias_rf['Importancia'],
         color='#27AE60', edgecolor='k')
plt.xlabel('Importancia', fontsize=12)
plt.title('Importancia de Características — Random Forest', fontweight='bold')
plt.grid(alpha=0.3, axis='x')
plt.show()

## 9. Ejercicios Propuestos

### Ejercicio 1: Árbol con criterio Entropía

Repite el análisis con `criterion='entropy'`. ¿Cambia el árbol resultante? ¿Qué variables son más importantes?

### Ejercicio 2: Poda agresiva

Prueba con `max_depth=2`. ¿Cuántas reglas tiene el árbol? ¿Qué accuracy pierdes?

### Ejercicio 3: Árbol como modelo interpretable

Entrena un árbol de profundidad 3 y escribe en español las reglas extraídas. ¿Tienen sentido pedagógico?

### Ejercicio 4: Random Forest optimizado

Usa `GridSearchCV` para optimizar `n_estimators`, `max_depth` y `min_samples_leaf` en Random Forest. ¿Cuánto puedes mejorar?

In [ ]:
# SOLUCIÓN Ejercicio 1: Árbol con Entropía
tree_entropy = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=4,
    class_weight='balanced',
    random_state=42
)
tree_entropy.fit(X_train, y_train)

print("="*80)
print("🔬 SOLUCIÓN Ejercicio 1: Árbol con Entropía")
print("="*80)
print(f"Accuracy Train: {tree_entropy.score(X_train, y_train)*100:.2f}%")
print(f"Accuracy Test:  {tree_entropy.score(X_test, y_test)*100:.2f}%")
print(f"Hojas: {tree_entropy.get_n_leaves()}")

# Comparar importancias
imp_entropy = pd.DataFrame({
    'Característica': feature_cols,
    'Importancia_Gini': tree_viz.feature_importances_,
    'Importancia_Entropy': tree_entropy.feature_importances_
}).sort_values('Importancia_Entropy', ascending=False)

print(f"\n📊 Comparación de importancias (Gini vs Entropía):")
print(imp_entropy.round(4).to_string(index=False))
print(f"\n💡 Ambos criterios identifican características similares como importantes")

## 10. Conclusiones

### Lecciones clave

1. **Árboles como baseline interpretable:** Un árbol con `max_depth=3` o `4` es fácilmente interpretable y proporciona reglas claras sobre qué variables predicen `Remontada`.

2. **La poda es esencial:** El árbol sin restricciones tiene un sobreajuste severo (100% en train, ~80% en test). La poda reduce la brecha a menos de 5%.

3. **Trade-off interpretabilidad-accuracy:** Los árboles simples son interpretables pero menos precisos. Random Forest mejora la precisión pero pierde interpretabilidad directa.

4. **Importancia de características:** El árbol revela que las variables **más importantes** son `Primer.Par_ed`, `Segundo.Par_ed` y `Firma_ed`.

5. **Comparación con otros modelos:** Random Forest supera a SVM y Regresión Logística en este dataset, pero SVM RBF sigue siendo competitivo.

### Sobre la variable 'Remontada'

- Es un problema **desbalanceado** (mayoría no remonta)
- Depende de la **evolución del desempeño** (Primer vs Segundo parcial)
- Requiere `class_weight='balanced'` para detectar bien la clase minoritaria
- La firma (compromiso) es un buen predictor de remontada

### Próximos pasos

1. **Boosting**: Gradient Boosting y XGBoost como mejora sobre Random Forest
2. **Feature Engineering**: crear variables como `mejora = Segundo.Par - Primer.Par`, `ratio_tplab`, etc.
3. **SHAP values**: interpretabilidad avanzada para modelos complejos
4. **Ensemble methods**: combinar árboles + SVM para predicción final